In [1]:
import sys
sys.path.append("src")
import retrieval as rl          # torch dimuat lebih dulu di dalamnya (macOS); jangan import faiss sebelum ini

res = rl.load_resources()       # chunks.json + index langchain + BM25


def tes(q, harap=None, metode=("bm25", "semantic", "hybrid", "gabung"), top_k=5,
        rerank=False, rerank_query_asli=False,
        rewrite=False, expand=False, variant="v1_fewshot", query_akhir=None, lihat=70):
    """Uji retrieval untuk satu pertanyaan yang kamu tulis sendiri.

    harap             nomor pasal yang kamu harapkan, mis. 476 atau [476, 477] -> hasil yang cocok diberi tanda ✓
    metode            bm25 | semantic | hybrid | gabung ("gabung" selalu memakai reranker)
    rerank            True = kandidat top_k*4 direrank ke top_k (untuk bm25/semantic/hybrid)
    rerank_query_asli True = reranker memakai pertanyaan asli, bukan query hasil rewrite/expand
    rewrite, expand   pakai LLM (butuh `ollama serve`); dijalankan SEKALI supaya semua metode memakai teks yang sama
    query_akhir       isi sendiri teks query yang mau diuji (melewati LLM), mis. hasil rewrite yang kamu tempel
    """
    harap = {harap} if isinstance(harap, int) else set(harap or [])

    nomor = rl.extract_pasal_numbers(q)
    if nomor:  # sama seperti pipeline: menyebut nomor pasal -> lookup langsung, tanpa retrieval
        print(f"Menyebut nomor pasal {nomor} -> lookup langsung (tanpa retrieval)")
        for n in nomor:
            d = res.pasal_index.get(n)
            print(f"  Pasal {n}: " + (d.page_content[:lihat].replace("\n", " ") + " ..." if d else "tidak ada di KUHP"))
        return

    if query_akhir:
        q_final = query_akhir
    elif rewrite or expand:
        q_final = rl.build_query_chain(rewrite, expand, variant).invoke(q)
    else:
        q_final = q

    print(f"Pertanyaan : {q}")
    if q_final != q:
        print(f"Query akhir: {q_final}")
    if harap:
        print(f"Diharapkan : pasal {sorted(harap)}")

    for m in metode:
        pakai_rerank = rerank or m == "gabung"
        if pakai_rerank:
            kandidat = rl.build_retriever(res, m, top_k * 4, use_reranker=False).invoke(q_final)
            kueri = q if rerank_query_asli else q_final
            docs = rl.CrossEncoderCompressor(scorer=rl.get_reranker(), top_n=top_k).compress_documents(kandidat, kueri)
            label = f"{m} + rerank" + (" (query asli)" if rerank_query_asli else "")
        else:
            docs = rl.build_retriever(res, m, top_k, use_reranker=False).invoke(q_final)
            label = m
        print(f"\n[{label}]")
        for i, d in enumerate(docs, 1):
            p = d.metadata["pasal"]
            tanda = "✓" if p in harap else " "
            print(f" {tanda}{i}. Pasal {p:>3} | skor {d.metadata['score']:8.3f} | {d.page_content[:lihat].replace(chr(10), ' ')}")
        if harap:
            ada = [d.metadata["pasal"] for d in docs if d.metadata["pasal"] in harap]
            print("   -> " + (f"ketemu: pasal {ada}" if ada else f"pasal {sorted(harap)} TIDAK ada di top-{top_k}"))

In [2]:
import json
from pathlib import Path

with open(Path("../Chatbot_KUHP/data/eval/pertanyaan.json"), encoding="utf-8") as f:   # sesuaikan path jika file ada di folder lain
    soal = json.load(f)

def jalankan(ids=None, **opsi):
    """Jalankan tes() untuk semua pertanyaan di pertanyaan.json, atau hanya yang id-nya ada di `ids`."""
    opsi = {"rewrite": True, "expand": True, "rerank": True, **opsi}   # bisa dioverride, mis. jalankan(rerank=False)
    for s in soal:
        if ids is not None and s["id"] not in ids:
            continue
        print("\n" + "=" * 80)
        print(f"{s['id']} | gaya: {s['gaya']}" + (" | AMBIGU" if s["ambigu"] else ""))
        print(f"Catatan    : {s['catatan']}")
        print("=" * 80)
        try:
            tes(s["pertanyaan"], harap=s["kandidat_pasal"], **opsi)
        except Exception as e:
            print(f"ERROR pada {s['id']}: {e}")

# jalankan(ids=["q01", "q02", "q03"])   # coba sebagian dulu
jalankan()                          # semua 64 pertanyaan


q01 | gaya: santai
Catatan    : Unsur utama: percobaan. Pasal pencurian (476/477) mungkin ikut relevan; ahli hukum menentukan.
Pertanyaan : belum sempat ngambil barangnya tapi udah keburu ketangkep pas lagi bongkar pintu, tetap bisa dipidana nggak?
Query akhir: Setiap Orang yang melakukan Percobaan untuk mengambil suatu Barang yang seluruhnya atau sebagian milik orang lain dengan cara merusak, memecahkan, atau membongkar
Diharapkan : pasal [17]


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


[bm25 + rerank]
  1. Pasal 521 | skor    3.926 | Pasal 521  (1) Setiap Orang yang secara melawan hukum merusak, menghan
  2. Pasal 476 | skor    3.421 | Pasal 476  Setiap Orang yang mengambil suatu Barang yang sebagian atau
  3. Pasal 479 | skor    3.343 | Pasal 479  (1) Setiap Orang yang melakukan pencurian yang didahului, d
  4. Pasal 581 | skor    2.990 | Pasal 581  Setiap Orang yang secara melawan hukum merusak, menghancurk
  5. Pasal 520 | skor    2.954 | Pasal 520  (1) Dipidana dengan pidana penjara paling lama 3 (tiga) tah
   -> pasal [17] TIDAK ada di top-5


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


[semantic + rerank]
  1. Pasal 476 | skor    3.421 | Pasal 476  Setiap Orang yang mengambil suatu Barang yang sebagian atau
  2. Pasal 479 | skor    3.343 | Pasal 479  (1) Setiap Orang yang melakukan pencurian yang didahului, d
  3. Pasal 581 | skor    2.990 | Pasal 581  Setiap Orang yang secara melawan hukum merusak, menghancurk
  4. Pasal 486 | skor    2.525 | Pasal 486 Setiap Orang yang secara melawan hukum memiliki suatu Barang
  5. Pasal 525 | skor    2.373 | Pasal 525 Setiap Orang yang secara melawan hukum menghancurkan atau me
   -> pasal [17] TIDAK ada di top-5

[hybrid + rerank]
  1. Pasal 521 | skor    3.926 | Pasal 521  (1) Setiap Orang yang secara melawan hukum merusak, menghan
  2. Pasal 476 | skor    3.421 | Pasal 476  Setiap Orang yang mengambil suatu Barang yang sebagian atau
  3. Pasal 479 | skor    3.343 | Pasal 479  (1) Setiap Orang yang melakukan pencurian yang didahului, d
  4. Pasal 581 | skor    2.990 | Pasal 581  Setiap Orang yang secara melawan hukum merusak, 